In [1]:
from _lib import *
from data import *

- load all the historical data and universe

In [2]:
boa = load_star_board()
tickers = boa["ticker"].tolist()
qt.log.info(f"no of tickers on STAR board - [{len(tickers)}]")

[JUSTY.LOG]	2026-05-10 00:59:53,435 - qt.common.help - INFO - no of tickers on STAR board - [609]


- rebalance parameters

In [3]:
eff_date = dt.date(2026, 6, 13)
annc_date = dt.date(2026, 5, 30)
cutoff_date = dt.date(2026, 4, 30)
hist_start = dt.date(2025, 5, 1)
cutoff_10 = (cutoff_date + pd.offsets.BDay(10)).date()
# sse_holidays = load_sse_holidays()

- get listing date

In [4]:
history = load_historical_data_ohlcv(tickers, hist_start, cutoff_date)

- add scraped data from sse

In [5]:
# merge info_df and tu on ticker
t_ld = get_listing_dates(tickers)

# add listing date to star board
boa = pd.merge(boa, t_ld, on="ticker", how="outer")

- special treatment securities

In [6]:
st_securities = boa[boa['st']]['ticker'].tolist()
qt.log.info(f"[{len(st_securities)}] ST securities: {st_securities}")

[JUSTY.LOG]	2026-05-10 00:59:59,230 - qt.common.help - INFO - [12] ST securities: ['688022', '688033', '688053', '688066', '688076', '688184', '688201', '688270', '688287', '688496', '688622', '688646']


- eligibility
	- Listing time > 6 months. 
		1. If no of securities listed > 12 months is b/w 100 to 150 then requirement changes to > 12 months
	- For securities with daily avg total market_cap since initial listing in top 5, listing time should be > 3 months as of 10th trading days after end date of data (cutoff date)
	- For securities with daily avg total market_cap since initial listing in top 3, listint time should be > 1 month
	- Non-* ST securities
	- No violation of laws/reg, no financial problems etc

In [7]:
history_after_cof = history.copy()

In [ ]:
univ = boa[[
	'ticker', 'listing_date', 'name', 'market_cap', 'st', 'shares_total', 
	'shares_tradable'
	]].copy()

univ['month_to_cof'] = univ['listing_date'].apply(lambda d: get_months_from_cutoff(d, cutoff=cutoff_date))
univ['month_to_cof10'] = univ['listing_date'].apply(lambda d: get_months_from_cutoff(d, cutoff=cutoff_10))

# add shares outstanding from yfinance
t_yf = get_yf_info(tickers=univ['ticker'].tolist())
univ = pd.merge(
	univ, 
	t_yf[[
		'ticker', 'shs_os_yf', 
		# 'shs_ff_yf'
		]], 
	on='ticker', how='left')

# if shs_os_yf is not null, use it as shs_total, otherwise use shares_total
# univ.loc[univ['shs_os_yf'] != 0, 'shs_for_avg'] = univ['shs_os_yf']
# univ.loc[univ['shs_os_yf'] == 0, 'shs_for_avg'] = univ['shares_total']
univ['shs_for_avg'] = univ['shares_tradable']

# add avg total mcap and avg value traded
avg_val_traded_n_mcap = get_avg_vol_mcap(history, univ, tickers=univ['ticker'].unique().tolist(), shs_col='shs_for_avg')

univ = pd.merge(univ, avg_val_traded_n_mcap, on='ticker', how='left')
univ['tmcap_rank'] = univ['avg_total_mcap'].rank(ascending=False, method='min')
univ['vtrad_rank'] = univ['avg_val_traded'].rank(ascending=False, method='min')

In [ ]:
if eff_date.month == 6:
	# read weight of existing index
	curr_s50 = load_star50_weights()

	# add weight column
	univ = pd.merge(univ, curr_s50[['ticker', 'curr_weight']], on='ticker', how='left')

elif eff_date.month == 3:
	qt.log.info(f"loading march rebal etf list")
	curr_s50 = load_star50_march_weights()
	univ['curr_weight'] = univ['ticker'].isin(curr_s50).astype(float)

- add eligibility

In [21]:
no_of_securities_mt_12m = len(univ[univ['month_to_cof'] >= 12])
listing_month_cutoff = 12 if no_of_securities_mt_12m > 100 else 6
qt.log.info(f"Listing month cutoff: {listing_month_cutoff} months (securities with month_to_cof >= {listing_month_cutoff}: {no_of_securities_mt_12m})")

univ['listing_elig'] = (
	((univ['tmcap_rank'] <= 3) & (univ['month_to_cof'] >= 1)) |
	((univ['tmcap_rank'] <= 5) & (univ['month_to_cof10'] >= 3)) |
	(univ['month_to_cof'] >= listing_month_cutoff)
)

# drop ST/*ST securities
st_univ = univ[univ['st']]
univ = univ[~univ['st']].reset_index(drop=True)

ineligible_univ = univ[~univ['listing_elig']]
univ = univ[univ['listing_elig']].reset_index(drop=True)
univ['vtrad_rank2'] = univ['avg_val_traded'].rank(ascending=False, method='min')

# drop 10% stocks based on avg value traded rank
low_val_traded_univ = univ[univ['vtrad_rank2'] > 0.9*len(univ)]
univ = univ[univ['vtrad_rank2'] <= 0.9*len(univ)].reset_index(drop=True)
univ['tmcap_rank2'] = univ['avg_total_mcap'].rank(ascending=False, method='min')
univ = univ.sort_values('tmcap_rank2').reset_index(drop=True)

[JUSTY.LOG]	2026-05-10 01:14:22,165 - qt.common.help - INFO - Listing month cutoff: 12 months (securities with month_to_cof >= 12: 502)


In [ ]:
# add ST, ineligible, low val traded univ back to the main univ for reference
univ = pd.concat([univ, st_univ, ineligible_univ, low_val_traded_univ], ignore_index=True)
univ = univ.sort_values(['tmcap_rank2', 'vtrad_rank2', 'listing_elig', 'st']).reset_index(drop=True)

In [11]:
# univ.set_index('ticker')[['shares_total', 'shares_tradable', 'shs_os_yf', 'shs_ff_yf']].plot()

In [20]:
# qt.view(univ)

In [ ]:
MAX_REPLACEMENTS = 5
comp_incl = univ[
	(univ['tmcap_rank2'] <= 40) &
	(univ['curr_weight'].isna())
][:MAX_REPLACEMENTS]
comp_excl = univ[
	(univ['tmcap_rank2'] > 60) &
	(~univ['curr_weight'].isna())
][:MAX_REPLACEMENTS]
print(f"inclusion stocks :")
qt.view2(comp_incl)
print(f"exclusion stocks :")
qt.view2(comp_excl)

inclusion stocks :


,ticker,listing_date,name,market_cap,st,shares_total,shares_tradable,month_to_cof,month_to_cof10,shs_os_yf,shs_for_avg,avg_total_mcap,avg_val_traded,count_data_pts,tmcap_rank,vtrad_rank,curr_weight,listing_elig,vtrad_rank2,tmcap_rank2
22,688498,2022-12-21,源杰科技,135.37,False,86.00,85.00,40,40,85.95,85.00,46.07,"2,104.42",242,24.00,13.00,NaN,True,10.00,23.00
28,688110,2021-12-10,东芯股份,68.47,False,442.00,442.00,52,53,442.25,442.00,41.25,"2,254.12",242,31.00,12.00,NaN,True,9.00,29.00
30,688002,2019-07-22,睿创微纳,68.58,False,466.00,466.00,81,81,465.74,466.00,39.97,615.03,242,33.00,88.00,NaN,True,74.00,31.00
34,688347,2023-08-07,华虹公司,276.21,False,"1,738.00",408.00,32,33,407.75,408.00,39.29,"2,089.95",232,37.00,14.00,NaN,True,11.00,35.00
37,688336,2020-07-22,三生国健,44.35,False,618.00,617.00,69,69,618.09,617.00,36.83,471.35,242,40.00,118.00,NaN,True,101.00,38.00
38,688585,2020-09-28,上纬新材,54.62,False,403.00,403.00,67,67,403.36,403.00,35.94,628.54,242,41.00,85.00,NaN,True,71.00,39.00


exclusion stocks :


,ticker,listing_date,name,market_cap,st,shares_total,shares_tradable,month_to_cof,month_to_cof10,shs_os_yf,shs_for_avg,avg_total_mcap,avg_val_traded,count_data_pts,tmcap_rank,vtrad_rank,curr_weight,listing_elig,vtrad_rank2,tmcap_rank2
66,688114,2022-09-09,华大智造,21.95,False,417.00,414.00,43,44,416.52,414.00,26.80,251.85,242,70.00,216.00,0.51,True,187.00,67.00
71,688702,2023-09-14,盛科通信,126.23,False,410.00,203.00,31,32,410.00,203.00,25.16,744.36,242,75.00,63.00,1.46,True,53.00,72.00
107,688472,2023-06-09,阿特斯,52.83,False,"3,643.00","1,348.00",34,35,"3,643.14","1,348.00",17.44,"1,082.15",242,113.00,32.00,0.83,True,24.00,108.00
131,688538,2021-05-28,和辉光电,32.31,False,"13,809.00","5,752.00",59,59,"13,809.44","5,752.00",14.69,200.20,242,139.00,263.00,0.50,True,232.00,132.00


- reporting and stuff

In [18]:
tmcap_40 = univ[univ['tmcap_rank2'] == 40]['avg_total_mcap'].values[0]
tmcap_60 = univ[univ['tmcap_rank2'] == 60]['avg_total_mcap'].values[0]
qt.log.info(f"TMCap of 40th stock: {tmcap_40:.3f} B CNY, TMCap of 60th stock: {tmcap_60:.3f} B CNY")

univ.loc[univ['curr_weight'].fillna(0) == 0, 'dist_to_40'] = univ['avg_total_mcap'].apply(lambda x: 100*(tmcap_40 - x)/tmcap_40)
univ.loc[univ['curr_weight'].fillna(0) != 0, 'dist_to_60'] = univ['avg_total_mcap'].apply(lambda x: 100*(tmcap_60 - x)/tmcap_60)

[JUSTY.LOG]	2026-05-10 01:10:19,721 - qt.common.help - INFO - TMCap of 40th stock: 35.484 B CNY, TMCap of 60th stock: 28.713 B CNY


In [19]:
univ

,ticker,listing_date,name,market_cap,st,shares_total,shares_tradable,month_to_cof,month_to_cof10,shs_os_yf,shs_for_avg,avg_total_mcap,avg_val_traded,count_data_pts,tmcap_rank,vtrad_rank,curr_weight,listing_elig,vtrad_rank2,tmcap_rank2,dist_to_40,dist_to_60
0,688041,2022-08-12,海光信息,751.41,False,"2,324.00","2,324.00",44,45,"2,324.34","2,324.00",472.51,"5,786.59",242,1.00,5.00,10.40,True,3.00,1.00,NaN,"-1,545.65"
1,688256,2020-07-20,DR寒武纪,742.98,False,628.00,628.00,69,69,628.31,628.00,457.83,"9,666.17",242,2.00,1.00,12.85,True,1.00,2.00,NaN,"-1,494.51"
2,688981,2020-07-16,中芯国际,964.06,False,"8,013.00","2,000.00",69,69,"1,999.56","2,000.00",213.34,"5,823.14",242,3.00,4.00,8.34,True,2.00,3.00,NaN,-643.02
3,688012,2019-07-22,中微公司,231.96,False,627.00,627.00,81,81,626.92,627.00,164.12,"3,160.17",242,4.00,8.00,5.62,True,6.00,4.00,NaN,-471.58
4,688008,2019-07-22,澜起科技,256.99,False,"1,222.00","1,146.00",81,81,"1,146.43","1,146.00",137.66,"4,685.52",242,5.00,6.00,8.35,True,4.00,5.00,NaN,-379.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,688623,2023-06-08,双元科技,5.80,False,59.00,19.00,34,35,59.14,19.00,1.50,61.18,242,582.00,513.00,NaN,True,475.00,498.00,95.77,NaN
498,688530,2024-05-09,欧莱新材,8.40,False,160.00,68.00,23,24,160.04,68.00,1.43,140.08,242,587.00,352.00,NaN,True,317.00,499.00,95.98,NaN
499,688479,2023-05-11,友车科技,6.14,False,144.00,62.00,35,36,144.32,62.00,1.39,48.41,242,588.00,539.00,NaN,True,500.00,500.00,96.10,NaN
500,688573,2023-08-17,信宇人,2.32,False,98.00,53.00,32,32,97.75,53.00,1.34,142.70,242,590.00,349.00,NaN,True,314.00,501.00,96.23,NaN


- march testing

In [23]:
# univ[univ['ticker'].isin(
# 	[
# 		'688498', '688110', '688002',
# 		'688114', '688278', '688349'	
#   	]
# )]

# excls = ['688220', '688301', '688385']
# incls = ['688213', '688278', '688578']
# res_list = ["688608", "688425", "688361", "688568", "688172",]
# univ[univ['ticker'].isin(excls)]
# univ[univ['ticker'].isin(incls)]
# univ[univ['ticker'].isin(res_list)]